# Laws-only Dense + Query Translation (NLLB 600M / 1.3B)

Same pipeline as `2026-03-07-helsinki.ipynb` but uses `facebook/nllb-200-distilled-600M` for EN→DE translation instead of Helsinki opus-mt-en-de.

**Why NLLB:** Higher-quality general translation (~2.5GB vs ~300MB). NLLB is a seq2seq model trained on 200 languages with explicit src/tgt language codes.

**Results so far:**
- NLLB 600M LB: **0.018** (worse than Helsinki 0.03 and even baseline 0.01816)

**Swap to 1.3B:** Just change `translation_model_name` to `facebook/nllb-200-1.3B`. Same API, same code.

In [ ]:
!pip install -q sentence_transformers

In [ ]:
import pandas as pd
import numpy as np
import time
import warnings
import gc
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

warnings.filterwarnings('ignore')

In [ ]:
BASE = '/kaggle/input/llm-agentic-legal-information-retrieval'

val_path    = f'{BASE}/val.csv'
test_path   = f'{BASE}/test.csv'
laws_path   = f'{BASE}/laws_de.csv'
output_path = 'submission.csv'

In [ ]:
embedding_model_name   = 'BAAI/bge-m3'
reranker_model_name    = 'BAAI/bge-reranker-v2-m3'
translation_model_name = 'facebook/nllb-200-distilled-600M'
src_lang               = 'eng_Latn'
tgt_lang               = 'deu_Latn'

use_gpu  = torch.cuda.is_available()
num_gpus = torch.cuda.device_count() if use_gpu else 0

if num_gpus >= 2:
    device_emb     = 'cuda:0'
    device_rerank  = 'cuda:1'
    device_transl  = 'cuda:1'
elif num_gpus == 1:
    device_emb     = 'cuda:0'
    device_rerank  = 'cuda:0'
    device_transl  = 'cuda:0'
else:
    device_emb     = 'cpu'
    device_rerank  = 'cpu'
    device_transl  = 'cpu'

print(f'devices — emb: {device_emb}, rerank: {device_rerank}, transl: {device_transl}')

laws_batch_size  = 8
top_k_retrieval  = 60
top_k_final      = 10
text_truncate    = 2000
transl_max_chars = 1500

In [ ]:
def load_data():
    val_df  = pd.read_csv(val_path)
    test_df = pd.read_csv(test_path)
    laws_df = pd.read_csv(laws_path)
    laws_df['text'] = laws_df['text'].fillna('')
    print(f'val: {len(val_df):,}  test: {len(test_df):,}  laws: {len(laws_df):,}')
    return val_df, test_df, laws_df

In [ ]:
def load_translator():
    print(f'loading translator ({translation_model_name}) on {device_transl}')
    tokenizer = AutoTokenizer.from_pretrained(translation_model_name)
    model     = AutoModelForSeq2SeqLM.from_pretrained(
        translation_model_name,
        torch_dtype=torch.float16
    ).to(device_transl)
    model.eval()
    print('translator ready')
    return tokenizer, model


def translate_to_german(text: str, tokenizer, model) -> str:
    """Translate English text to German using NLLB. Chunks long inputs."""
    chunks, current = [], []
    current_len = 0
    for para in text.split('\n'):
        if current_len + len(para) > transl_max_chars and current:
            chunks.append('\n'.join(current))
            current, current_len = [], 0
        current.append(para)
        current_len += len(para)
    if current:
        chunks.append('\n'.join(current))

    translated_chunks = []
    for chunk in chunks:
        if not chunk.strip():
            continue
        inputs = tokenizer(
            chunk, return_tensors='pt',
            padding=True, truncation=True, max_length=512,
            src_lang=src_lang
        ).to(device_transl)
        forced_bos = tokenizer.convert_tokens_to_ids(tgt_lang)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos,
                max_new_tokens=512
            )
        translated_chunks.append(tokenizer.decode(output_ids[0], skip_special_tokens=True))

    return '\n'.join(translated_chunks)

In [ ]:
def load_embedding_model():
    print(f'loading embedder on {device_emb}')
    model = SentenceTransformer(
        embedding_model_name,
        device=device_emb,
        model_kwargs={'torch_dtype': torch.float16}
    )
    print('embedder ready')
    return model


def encode_corpus(model, texts):
    truncated = [str(t)[:text_truncate] for t in texts]
    return model.encode(
        truncated,
        batch_size=laws_batch_size,
        convert_to_tensor=False,
        normalize_embeddings=True,
        show_progress_bar=True
    )


def encode_query(model, query):
    return model.encode(
        query,
        convert_to_tensor=True,
        normalize_embeddings=True
    ).cpu().numpy()


def search_dense(query_embedding, corpus_embeddings, top_k):
    scores      = np.dot(corpus_embeddings, query_embedding)
    top_indices = np.argsort(scores)[-top_k:][::-1]
    return [(int(idx), float(scores[idx])) for idx in top_indices]

In [ ]:
def load_reranker():
    print(f'loading reranker on {device_rerank}')
    model = CrossEncoder(
        reranker_model_name,
        device=device_rerank,
        max_length=512,
        model_kwargs={'torch_dtype': torch.float16}
    )
    print('reranker ready')
    return model


def rerank_candidates(reranker, query_de: str, candidates: list) -> list:
    if not candidates:
        return []
    pairs     = [[query_de, str(text)[:text_truncate]] for _, text, _ in candidates]
    citations = [citation for citation, _, _ in candidates]
    scores    = reranker.predict(pairs, batch_size=8, show_progress_bar=False)
    ranked    = sorted(zip(citations, scores), key=lambda x: x[1], reverse=True)
    return [c for c, _ in ranked]

In [ ]:
def retrieve_citations(
    query_emb: np.ndarray,
    query_de: str,
    laws_embeddings,
    laws_df,
    reranker,
) -> list:
    dense_results = search_dense(query_emb, laws_embeddings, top_k_retrieval)
    candidates = [
        (laws_df.iloc[idx]['citation'], laws_df.iloc[idx]['text'], score)
        for idx, score in dense_results
        if 0 <= idx < len(laws_df)
    ]
    ranked = rerank_candidates(reranker, query_de, candidates)
    return ranked[:top_k_final]

In [ ]:
def main():
    start = time.time()

    val_df, test_df, laws_df = load_data()
    queries_en = test_df['query'].astype(str).tolist()

    # --- Phase 1: translate all queries EN→DE, then free translator ---
    transl_tokenizer, transl_model = load_translator()
    print(f'translating {len(queries_en)} queries...')
    queries_de = [translate_to_german(q, transl_tokenizer, transl_model) for q in queries_en]
    del transl_model, transl_tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print('translations done')

    # --- Phase 2: encode corpus + all queries, then free embedder ---
    embedding_model = load_embedding_model()
    print('encoding laws corpus...')
    laws_embeddings = encode_corpus(embedding_model, laws_df['text'].tolist())
    print(f'  done — shape {laws_embeddings.shape}')
    print('encoding queries...')
    query_embeddings = [encode_query(embedding_model, q) for q in queries_en]
    del embedding_model
    gc.collect()
    torch.cuda.empty_cache()
    print('embeddings done')

    # --- Phase 3: rerank, then free reranker ---
    reranker = load_reranker()
    print(f'reranking {len(test_df)} test queries...')
    predictions = []
    for i, (query_emb, query_de) in enumerate(zip(query_embeddings, queries_de)):
        citations = retrieve_citations(query_emb, query_de, laws_embeddings, laws_df, reranker)
        predictions.append(citations)
        if (i + 1) % 10 == 0:
            print(f'  {i+1}/{len(test_df)}')
    del reranker
    gc.collect()
    torch.cuda.empty_cache()

    submission_df = pd.DataFrame({
        'query_id':            test_df['query_id'],
        'predicted_citations': [';'.join(p) for p in predictions]
    })
    submission_df.to_csv(output_path, index=False)
    print(f'saved {output_path}  ({time.time()-start:.0f}s)')
    return submission_df


submission_df = main()